# Kaggle Competition sklearn Pipeline Skeleton


In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression


In [ ]:
# Load Data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(train.shape)
print(test.shape)


In [ ]:
# EDA
print(train.head())
print(train.dtypes)
print(train.describe())
print(train.isnull().sum())


In [ ]:
TARGET = 'target'  # replace with actual target column
X = train.drop(columns=[TARGET])
y = train[TARGET]
X_test = test.copy()


In [ ]:
# Missing Value Handling
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())
    X_test[col] = X_test[col].fillna(X[col].median())
for col in cat_cols:
    mode_val = X[col].mode()[0]
    X[col] = X[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)


In [ ]:
# Label Encoding
X_label = X.copy()
X_test_label = X_test.copy()
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_label[col], X_test_label[col]])
    le.fit(combined)
    X_label[col] = le.transform(X_label[col])
    X_test_label[col] = le.transform(X_test_label[col])


In [ ]:
# Ordinal Encoding
X_ord = X.copy()
X_test_ord = X_test.copy()
if len(cat_cols) > 0:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_ord[cat_cols] = oe.fit_transform(X_ord[cat_cols])
    X_test_ord[cat_cols] = oe.transform(X_test_ord[cat_cols])


In [ ]:
# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_ord)
X_test_scaled = scaler.transform(X_test_ord)


In [ ]:
# Logistic Regression
log_model = LogisticRegression(max_iter=1000)
scores = cross_val_score(log_model, X_scaled, y, cv=5, scoring='accuracy')
print(scores.mean())


In [ ]:
# Linear Regression
lin_model = LinearRegression()
scores = cross_val_score(lin_model, X_scaled, y, cv=5, scoring='r2')
print(scores.mean())


In [ ]:
# Submission File
log_model.fit(X_scaled, y)
preds = log_model.predict(X_test_scaled)
submission = pd.DataFrame({'id': test['id'], TARGET: preds})
submission.to_csv('submission.csv', index=False)
